# cloudposterior: Phone Notifications

Monitor MCMC sampling from your phone using [ntfy](https://ntfy.sh) push notifications. Progress updates live with per-chain stats, divergence counts, and speed.

> Run this notebook locally to see the QR code and notification link.

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm

import cloudposterior as cp

In [ ]:
df = pd.read_csv(pm.get_data("radon.csv"))

with pm.Model(name="radon_intercepts", coords={"county": df.county.unique()}) as radon:
    mu_a = pm.Normal("mu_a", mu=0, sigma=5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    a_raw = pm.Normal("a_raw", mu=0, sigma=1, dims="county")
    a = pm.Deterministic("a", mu_a + sigma_a * a_raw, dims="county")
    b_floor = pm.Normal("b_floor", mu=0, sigma=5)
    mu = a[df.county_code.values] + b_floor * df.floor.values
    sigma_y = pm.HalfNormal("sigma_y", sigma=2)
    pm.Normal("obs", mu=mu, sigma=sigma_y, observed=df.log_radon.values)

## Basic notifications

Pass `notify=True` to get a unique ntfy.sh topic. A link and QR code are printed -- scan it to subscribe on your phone. Notifications update in place as sampling progresses.

For private notifications, you can point to your own [ntfy server](https://docs.ntfy.sh/install/) with `notify={"server": "https://ntfy.example.com"}`.

In [3]:
with cp.cloud(radon, cache="disk", notify=True):
    idata = pm.sample(draws=100000, tune=1000, chains=4)

We recommend running at least 4 chains for robust computation of convergence diagnostics


## Custom topic

Use a fixed topic name so you can subscribe once and reuse it across runs. Anyone with the topic name can see your notifications -- pick something unique.

In [ ]:
with cp.cloud(radon, cache="disk", notify="my-radon-project"):
    idata = pm.sample(draws=2000, tune=1000, chains=4)